In [ ]:
!pip install transformers datasets torch scikit-learn -q
print("Done")

Done


In [ ]:
import random
import pandas as pd
import numpy as np

random.seed(42)
np.random.seed(42)

positive_reviews = [
    "Food was absolutely delicious and delivery was super fast!",
    "Amazing taste, fresh ingredients, will order again.",
    "Best biryani I have ever had, highly recommended.",
    "Packaging was great and food arrived hot.",
    "Excellent service and the pizza was perfect.",
    "Very tasty food, good portions and quick delivery.",
    "Loved the samosas, crispy and perfectly spiced.",
    "Great experience overall, food quality is top notch.",
    "Delivery was on time and food was outstanding.",
    "Freshly prepared meal, exceeded my expectations.",
    "The sushi was fresh and beautifully presented.",
    "Fantastic flavors, will definitely order again.",
    "Quick delivery and the food was still hot.",
    "Best restaurant on the app, never disappoints.",
    "Wonderful meal, generous portions and great taste.",
]

negative_reviews = [
    "Food arrived cold and was completely tasteless.",
    "Very late delivery, food was soggy and bad.",
    "Worst experience ever, never ordering from here again.",
    "Packaging was torn and food was spilled inside.",
    "Stale food delivered, made me sick afterwards.",
    "Extremely disappointed with the quality of food.",
    "Order was wrong and customer support was useless.",
    "Very overpriced for such poor quality food.",
    "Food smelled bad and tasted even worse.",
    "Delivery took 2 hours and food was cold.",
    "Missing items in order, no refund provided.",
    "Terrible taste, nothing like what was shown in photo.",
    "Driver was rude and food was dropped at wrong address.",
    "Horrible experience, food was undercooked.",
    "Never again, complete waste of money.",
]

neutral_reviews = [
    "Food was okay, nothing special but edible.",
    "Average experience, delivery was on time.",
    "Decent food but portion size could be better.",
    "Neither good nor bad, just regular food.",
    "Okay for the price, would not strongly recommend.",
    "Food was acceptable, packaging was average.",
    "Standard delivery, food was as expected.",
    "Not bad but not great either.",
    "Mediocre food but arrived on time.",
    "Average quality, might order again if nothing better.",
]

reviews, labels = [], []

for _ in range(600):
    reviews.append(random.choice(positive_reviews) + " " + random.choice(positive_reviews))
    labels.append(2)  # positive

for _ in range(600):
    reviews.append(random.choice(negative_reviews) + " " + random.choice(negative_reviews))
    labels.append(0)  # negative

for _ in range(400):
    reviews.append(random.choice(neutral_reviews) + " " + random.choice(neutral_reviews))
    labels.append(1)  # neutral

df = pd.DataFrame({"review": reviews, "sentiment": labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total reviews: {len(df)}")
print(f"Sentiment distribution:\n{df['sentiment'].value_counts().sort_index()}")
print(f"\nSample:\n{df.head(3)}")

Total reviews: 1600
Sentiment distribution:
sentiment
0    600
1    400
2    600
Name: count, dtype: int64

Sample:
                                              review  sentiment
0  The sushi was fresh and beautifully presented....          2
1  Freshly prepared meal, exceeded my expectation...          2
2  Very tasty food, good portions and quick deliv...          2


In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torch

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sentiment'])
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = ReviewDataset(train_df['review'], train_df['sentiment'], tokenizer)
test_dataset  = ReviewDataset(test_df['review'],  test_df['sentiment'],  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print("Tokenization complete. DataLoaders ready.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train: 1280 | Test: 320
Tokenization complete. DataLoaders ready.


In [ ]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
EPOCHS = 5

print(f"Device: {DEVICE}")

# Load pretrained DistilBERT with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
)
model = model.to(DEVICE)

optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()


def train_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    return 100. * correct / total


print("Training DistilBERT for sentiment analysis...\n")
best_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader)
    test_acc = evaluate(model, test_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

    if test_acc > best_acc:
        best_acc = test_acc
        model.save_pretrained("sentiment_model")
        tokenizer.save_pretrained("sentiment_model")

print(f"\nBest Accuracy: {best_acc:.2f}%")
print("Model saved to sentiment_model/")

Device: cuda


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training DistilBERT for sentiment analysis...

Epoch [1/5] | Loss: 0.4894 | Train Acc: 86.33% | Test Acc: 100.00%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch [2/5] | Loss: 0.0370 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch [3/5] | Loss: 0.0142 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch [4/5] | Loss: 0.0085 | Train Acc: 100.00% | Test Acc: 100.00%
Epoch [5/5] | Loss: 0.0058 | Train Acc: 100.00% | Test Acc: 100.00%

Best Accuracy: 100.00%
Model saved to sentiment_model/


In [ ]:
LABEL_NAMES = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"}

def predict_sentiment(text):
    model.eval()
    encoding = tokenizer(
        text,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = outputs.logits.argmax(dim=1).item()

    return LABEL_NAMES[pred]


# Test with real-world style reviews
test_reviews = [
    "Food was amazing and delivery was super fast, loved it!",
    "Terrible experience, food was cold and tasted awful.",
    "Okay food, nothing special but it was fine.",
    "Best pizza I have ever had, will definitely order again!",
    "Very late delivery and the order was completely wrong.",
]

print("NourishNet AI - Sentiment Analysis Results")
print("-" * 50)
for review in test_reviews:
    sentiment = predict_sentiment(review)
    print(f"Review : {review[:55]}...")
    print(f"Sentiment: {sentiment}\n")

NourishNet AI - Sentiment Analysis Results
--------------------------------------------------
Review : Food was amazing and delivery was super fast, loved it!...
Sentiment: POSITIVE

Review : Terrible experience, food was cold and tasted awful....
Sentiment: NEGATIVE

Review : Okay food, nothing special but it was fine....
Sentiment: NEUTRAL

Review : Best pizza I have ever had, will definitely order again...
Sentiment: POSITIVE

Review : Very late delivery and the order was completely wrong....
Sentiment: NEGATIVE



In [ ]:
import shutil
from google.colab import files

# Zip the model folder
shutil.make_archive("sentiment_model", "zip", "sentiment_model")
files.download("sentiment_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>